In [ ]:
import os
import pandas as pd
import numpy as np
from plotly import graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import to_html
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


%load_ext autoreload
%autoreload 2


In [ ]:
# Load pile_aggregate and eval_aggregate data
# pile_losses = pd.read_csv('pile_aggregate_n44.csv', index_col=0)
# eval_losses = pd.read_csv('evals_aggregate_n60.csv', index_col=0)
pile_losses = pd.read_csv('pile_aggregate_n58.csv', index_col=0)
eval_losses = pd.read_csv('evals_aggregate_n58.csv', index_col=0)
print(f"Original Pile losses shape: {pile_losses.shape}")
print(f"Original Eval losses shape: {eval_losses.shape}")

# Restrict both dfs to the same rows
common_indexes = pile_losses.index.intersection(eval_losses.index)
pile_losses = pile_losses.loc[common_indexes]
eval_losses = eval_losses.loc[common_indexes]
print(f"Pile losses shape: {pile_losses.shape}")
print(f"Eval losses shape: {eval_losses.shape}")

figure_path = "figures/aggregate_pca_v2"

In [ ]:
pile_small = pd.read_csv('pile_aggregate_n44.csv', index_col=0)
# which indexes of pile_losses are not in pile_small?
missing_indexes = pile_losses.index.difference(pile_small.index)
print(missing_indexes)

# PCA on standardised, full-model (n=58) X and Y

In [ ]:
from aggregate_pca import PCAAnalysis
standardise = True
n_components = 10
pca_X = PCAAnalysis(pile_losses, name="Pile", standardise=standardise, n_components=n_components)
pca_Y = PCAAnalysis(eval_losses, name="Evals", standardise=standardise, n_components=n_components)

In [ ]:
from aggregate_pca import plot_pc_score_heatmaps
fig1 = plot_pc_score_heatmaps(pca_X, pca_Y)
fig1.show()
fig1.write_image(f"{figure_path}/pc_score_heatmaps_n{len(pca_X.scores.index)}.png", scale=3)
fig1.write_html(f"{figure_path}/pc_score_heatmaps_n{len(pca_X.scores.index)}.html")

### NOTE: SV Scaled Score means U_X S_X (i.e. scaled by their singular values [SV]) whereas Unit Scores means just U_X.

In [ ]:
from aggregate_pca import plot_score_correlation_heatmap

fig_corr = plot_score_correlation_heatmap(pca_X, pca_Y)
fig_corr.show()
fig_corr.write_image(f"{figure_path}/pc_score_correlations_n{len(pca_X.scores.index)}.png", scale=3)
fig_corr.write_html(f"{figure_path}/pc_score_correlations_n{len(pca_X.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig2 = plot_pc_loading_heatmap(pca_Y)
fig2.show()
fig2.write_image(f"{figure_path}/pc_eval_loadings_n{len(pca_Y.scores.index)}.png", scale=3)
fig2.write_html(f"{figure_path}/pc_eval_loadings_n{len(pca_Y.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig3 = plot_pc_loading_heatmap(pca_X)
fig3.show()
fig3.write_image(f"{figure_path}/pc_pile_loadings_contexts_n{len(pca_X.scores.index)}.png", scale=2)
fig3.write_html(f"{figure_path}/pc_pile_loadings_contexts_n{len(pca_X.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig4 = plot_pc_loading_heatmap(pca_X, mean_loadings=True)
fig4.show()
fig4.write_image(f"{figure_path}/pc_pile_loadings_means_n{len(pca_X.scores.index)}.png", scale=3)
fig4.write_html(f"{figure_path}/pc_pile_loadings_means_n{len(pca_X.scores.index)}.html")

In [ ]:
from aggregate_pca import calculate_dk_values, plot_dk_dict, calculate_dk_dict
max_k_X = 15
dk_values = calculate_dk_dict(pca_X, pca_Y, standardise=True, max_k_X=max_k_X)
fig_dk = plot_dk_dict(dk_values, pca_X_name="Pile", pca_Y_name="Evals", max_k_Y=None)
fig_dk.show()
fig_dk.write_image(f"{figure_path}/Dk_maxkx{max_k_X}_n{len(pca_X.scores.index)}.png", scale=3)
fig_dk.write_html(f"{figure_path}/Dk_maxkx{max_k_X}_n{len(pca_X.scores.index)}.html")

# PCA on standardised X and Y with No Outliers (NO) (n=52)

In [ ]:
from aggregate_pca import PCAAnalysis
standardise = True
n_components = 10
# Also apply PCA to both datasets without outliers (NO= No Outliers)
outlier_models = ['eleutherai/pythia-14m','eleutherai/pythia-31m',
                  'eleutherai/pythia-70m','eleutherai/pythia-160m', 
                  'eleutherai/pythia-410m',
                  'google/gemma-7b-it', 'google/gemma-2b-it']
pca_X_NO = PCAAnalysis(pile_losses[~pile_losses.index.isin(outlier_models)], name="Pile (NO)", 
                       standardise=standardise, n_components=n_components)
pca_Y_NO = PCAAnalysis(eval_losses[~eval_losses.index.isin(outlier_models)], name="Evals (NO)", 
                       standardise=standardise, n_components=n_components)

In [ ]:
from aggregate_pca import plot_pc_score_heatmaps
fig1 = plot_pc_score_heatmaps(pca_X_NO, pca_Y_NO)
fig1.show()
fig1.write_image(f"{figure_path}/pc_score_heatmaps_NO_n{len(pca_X_NO.scores.index)}.png", scale=3)
fig1.write_html(f"{figure_path}/pc_score_heatmaps_NO_n{len(pca_X_NO.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_score_correlation_heatmap

fig_corr = plot_score_correlation_heatmap(pca_X_NO, pca_Y_NO)
fig_corr.show()
fig_corr.write_image(f"{figure_path}/pc_score_correlations_NO_n{len(pca_X_NO.scores.index)}.png", scale=3)
fig_corr.write_html(f"{figure_path}/pc_score_correlations_NO_n{len(pca_X_NO.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig2 = plot_pc_loading_heatmap(pca_Y_NO)
fig2.show()
fig2.write_image(f"{figure_path}/pc_eval_loadings_NO_n{len(pca_X_NO.scores.index)}.png", scale=3)
fig2.write_html(f"{figure_path}/pc_eval_loadings_NO_n{len(pca_X_NO.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig3 = plot_pc_loading_heatmap(pca_X_NO)
fig3.show()
fig3.write_image(f"{figure_path}/pc_pile_loadings_contexts_NO_n{len(pca_X_NO.scores.index)}.png", scale=2)
fig3.write_html(f"{figure_path}/pc_pile_loadings_contexts_NO_n{len(pca_X_NO.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_pc_loading_heatmap
fig4 = plot_pc_loading_heatmap(pca_X_NO, mean_loadings=True)
fig4.show()
fig4.write_image(f"{figure_path}/pc_pile_loadings_means_NO_n{len(pca_X_NO.scores.index)}.png", scale=3)
fig4.write_html(f"{figure_path}/pc_pile_loadings_means_NO_n{len(pca_X_NO.scores.index)}.html")

In [ ]:
from aggregate_pca import calculate_dk_values, plot_dk_dict, calculate_dk_dict
max_k_X = 15
dk_values = calculate_dk_dict(pca_X_NO, pca_Y_NO, standardise=True, max_k_X=max_k_X)
fig_dk = plot_dk_dict(dk_values, pca_X_name="Pile (NO)", pca_Y_name="Evals (NO)")
fig_dk.show()
fig_dk.write_image(f"{figure_path}/Dk_maxkx{max_k_X}_NO_n{len(pca_X_NO.scores.index)}.png", scale=3)
fig_dk.write_html(f"{figure_path}/Dk_maxkx{max_k_X}_NO_n{len(pca_X_NO.scores.index)}.html")

# Plot raw data

In [ ]:
from aggregate_pca import plot_raw_data_heatmap
fig = plot_raw_data_heatmap(pca_Y, sort_by='PC1', ascending=False, 
                            standardse=True)
fig.show()
fig.write_image(f"{figure_path}/raw_standardised_data_eval_heatmap_n{len(pca_X.scores.index)}.png", scale=3)
fig.write_html(f"{figure_path}/raw_standardised_data_eval_heatmap_n{len(pca_X.scores.index)}.html")

### NOTE - These are the standardised scores, which are what are being fed into the PCA above. 
# For raw scores, set standardse=False in the plot_raw_data_heatmap function.

In [ ]:
from aggregate_pca import plot_raw_data_heatmap
fig = plot_raw_data_heatmap(pca_X, sort_by='PC1', ascending=True, 
                            standardise=True, negate_values=True)
fig.show()
fig.write_image(f"{figure_path}/raw_standardised_data_pile_heatmap_n{len(pca_X.scores.index)}.png", scale=2)
fig.write_html(f"{figure_path}/raw_standardised_data_pile_heatmap_n{len(pca_X.scores.index)}.html")

In [ ]:
from aggregate_pca import plot_raw_data_heatmap
fig = plot_raw_data_heatmap(pca_X_NO, sort_by='PC1', ascending=True, 
                            standardise=True, negate_values=True)
fig.show()
fig.write_image(f"{figure_path}/raw_standardised_data_pile_NO_heatmap.png", scale=2)
fig.write_html(f"{figure_path}/raw_standardised_data_pile_NO_heatmap.html")

# Null hypothesis testing

In [ ]:
from aggregate_pca import run_null_hypothesis_experiment, calculate_dk_dict, plot_dk_with_null_comparison

null_stats = run_null_hypothesis_experiment(
    pca_X, pca_Y, 
    max_k_X=12, 
    standardise=True, 
    n_trials=100,
    permute_Y=False  # Set to True if you want to permute Y as well
)

# Calculate observed D(k) values
observed_dk_dict = calculate_dk_dict(pca_X, pca_Y, max_k_X=12, standardise=True)


In [ ]:
fig = plot_dk_with_null_comparison(observed_dk_dict, null_stats, pca_X_name="X", pca_Y_name="Y", max_k_Y=None)
fig.show()
fig.write_image(f"{figure_path}/Dk_with_null_comparison_n{len(pca_X.scores.index)}.png", scale=3)
fig.write_html(f"{figure_path}/Dk_with_null_comparison_n{len(pca_X.scores.index)}.html")

# Comparing distribution in two variables (cols of X or Y)

In [ ]:
from aggregate_pca import plot_marginal_pdf
fig = plot_marginal_pdf(pca_Y, x_col='gsm8k', y_col='mathqa')
fig.show()
fig.write_image(f"{figure_path}/raw_eval_data_marginal_pdfs_gsm8k_vs_mathqa_n{len(pca_Y.scores.index)}.png", scale=3)
fig.write_html(f"{figure_path}/raw_eval_data_marginal_pdfs_gsm8k_vs_mathqa_n{len(pca_Y.scores.index)}.html")

# X-PC linear regression reconstruction error heatmap

In [ ]:
from aggregate_pca import analyse_reconstruction_errors
reconstruction_results = analyse_reconstruction_errors(pca_X, pca_Y, standardise=True)

In [ ]:
from aggregate_pca import plot_reconstruction_errors_with_slider
fig = plot_reconstruction_errors_with_slider(
    reconstruction_results, 
    hparam_name="Latent factors of X (k_X)",
    sorted_evals=pca_Y.scores.sort_values(ascending=False, by='PC1').index,
    sorted_models=pca_Y.loadings.sort_values(ascending=True, by='PC1').index
)

fig.show()
fig.write_html(f"{figure_path}/reconstruction_errors_Y_from_X-PCs_n{len(pca_X.scores.index)}.html")
fig.write_image(f"{figure_path}/reconstruction_errors_Y_from_X-PCs_n{len(pca_X.scores.index)}.png", scale=3)